In [2]:
# @title Step 1: Verify GPU
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
print()

import torch
print(f"  PyTorch {torch.__version__}")
print(f"  CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    print(f"  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

name, memory.total [MiB], driver_version
Tesla T4, 15360 MiB, 580.105.08
Tesla T4, 15360 MiB, 580.105.08

  PyTorch 2.10.0+cu128
  CUDA available: True
  GPU: Tesla T4
  VRAM: 14.6 GB


In [3]:
# @title Step 2: Install uv
import sys
print(f"Python: {sys.version}")
assert sys.version_info >= (3, 11), "ACE-Step 1.5 requires Python >= 3.11"

print("\nInstalling uv package manager...")
!curl -LsSf https://astral.sh/uv/install.sh | sh 2>&1 | tail -1

import os
os.environ["PATH"] = os.path.expanduser("~/.local/bin") + ":" + os.environ["PATH"]
!uv --version
print("Done!")

Python: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]

Installing uv package manager...
everything's installed!
uv 0.11.5 (x86_64-unknown-linux-gnu)
Done!


In [4]:
# @title Step 3: Clone & Install
%cd /content

import os
if os.path.exists("/content/ACE-Step-1.5"):
    print("Repository exists, pulling latest...")
    %cd /content/ACE-Step-1.5
    !git pull
else:
    print("Cloning ACE-Step 1.5...")
    !git clone https://github.com/ACE-Step/ACE-Step-1.5.git
    %cd /content/ACE-Step-1.5

print("\nInstalling dependencies (this may take 2-3 minutes)...\n")
!uv sync

print("\nAll dependencies installed!")

/content
Cloning ACE-Step 1.5...
Cloning into 'ACE-Step-1.5'...
remote: Enumerating objects: 11145, done.
remote: Counting objects: 100% (3547/3547), done.
remote: Compressing objects: 100% (831/831), done.
remote: Total 11145 (delta 3110), reused 2716 (delta 2716), pack-reused 7598 (from 2)
Receiving objects: 100% (11145/11145), 12.54 MiB | 22.14 MiB/s, done.
Resolving deltas: 100% (6985/6985), done.
/content/ACE-Step-1.5

Installing dependencies (this may take 2-3 minutes)...

Using CPython 3.12.12 interpreter at: /usr/bin/python3
Creating virtual environment at: .venv
Resolved 176 packages in 10.03s                                      
Prepared 139 packages in 1m 11s                                          
Installed 139 packages in 548ms                             
 + absl-py==2.4.0
 + accelerate==1.13.0
 + ace-step==1.5.0 (from file:///content/ACE-Step-1.5)
 + aiofiles==24.1.0
 + aiohappyeyeballs==2.6.1
 + aiohttp==3.13.5
 + aiosignal==1.4.0
 + annotated-doc==0.0.4
 + annotated

In [5]:
import os

# Temporary Fix RuntimeError: Generation produced NaN or Inf latents
# https://github.com/ace-step/ACE-Step-1.5/issues/1055
file_path = '/content/ACE-Step-1.5/acestep/core/generation/handler/init_service_orchestrator.py'

patched_code = """\"\"\"Top-level initialization orchestration for the handler.\"\"\"

import os
import traceback
from pathlib import Path
from typing import Optional, Tuple

import torch
from loguru import logger

from acestep import gpu_config

_ROCM_DTYPE_MAP = {
    "float32": torch.float32,
    "float16": torch.float16,
    "bfloat16": torch.bfloat16,
}


def _cuda_supports_bfloat16() -> bool:
    \"\"\"Return whether the active CUDA device supports native bfloat16 kernels.\"\"\"
    return gpu_config.cuda_supports_bfloat16()


def _resolve_rocm_dtype() -> torch.dtype:
    \"\"\"Return a safe model dtype for ROCm/HIP devices.

    Uses ``float32`` by default to avoid segfaults from incomplete
    ``bfloat16`` kernel support on some ROCm GPU configurations (e.g.
    AMD iGPUs on Strix Halo).  Set the ``ACESTEP_ROCM_DTYPE`` environment
    variable to ``float16`` or ``bfloat16`` to override for hardware that
    fully supports those formats.
    \"\"\"
    raw = os.environ.get("ACESTEP_ROCM_DTYPE", "float32").strip().lower()
    dtype = _ROCM_DTYPE_MAP.get(raw)
    if dtype is None:
        logger.warning(
            f"[initialize_service] Unknown ACESTEP_ROCM_DTYPE={raw!r}; "
            "falling back to float32."
        )
        dtype = torch.float32
    return dtype


class InitServiceOrchestratorMixin:
    \"\"\"Public ``initialize_service`` orchestration entrypoint.\"\"\"

    def initialize_service(
        self,
        project_root: str,
        config_path: str,
        device: str = "auto",
        use_flash_attention: bool = False,
        compile_model: bool = False,
        offload_to_cpu: bool = False,
        offload_dit_to_cpu: bool = False,
        quantization: Optional[str] = None,
        prefer_source: Optional[str] = None,
        use_mlx_dit: bool = True,
    ) -> Tuple[str, bool]:
        \"\"\"Initialize model artifacts and runtime backends for generation.

        This method intentionally supports repeated calls to reinitialize models
        with new settings; it does not short-circuit when components are already loaded.
        \"\"\"
        try:
            if config_path is None:
                config_path = "acestep-v15-turbo"
                logger.warning(
                    "[initialize_service] config_path not set; defaulting to 'acestep-v15-turbo'."
                )

            resolved_device = self._resolve_initialize_device(device)
            self.device = resolved_device
            self.offload_to_cpu = offload_to_cpu
            self.offload_dit_to_cpu = offload_dit_to_cpu

            normalized_compile, normalized_quantization, mlx_compile_requested = self._configure_initialize_runtime(
                device=resolved_device,
                compile_model=compile_model,
                quantization=quantization,
            )
            self.compiled = normalized_compile
            
            if resolved_device == "cuda" and gpu_config.is_rocm_available():
                self.dtype = _resolve_rocm_dtype()
                logger.info(
                    f"[initialize_service] ROCm/HIP device detected: using dtype={self.dtype} "
                    "(set ACESTEP_ROCM_DTYPE=bfloat16 or float16 to override)"
                )
            # --- PATCH APPLIED HERE ---
            elif resolved_device == "cuda":
                import os
                _forced_dtype = os.environ.get("ACESTEP_DTYPE", "").lower()
                if _forced_dtype == "float32":
                    self.dtype = torch.float32
                    logger.info("[initialize_service] ACESTEP_DTYPE=float32 override applied.")
                elif gpu_config.cuda_supports_bfloat16():
                    self.dtype = torch.bfloat16
                else:
                    self.dtype = torch.float16
                    logger.info(
                        "[initialize_service] Pre-Ampere CUDA detected: "
                        "using float16 instead of bfloat16."
                    )
            # --------------------------
            else:
                self.dtype = torch.bfloat16 if resolved_device == "xpu" else torch.float32
                
            self.quantization = normalized_quantization
            try:
                self._validate_quantization_setup(
                    quantization=self.quantization,
                    compile_model=normalized_compile,
                )
            except ImportError as exc:
                if self.quantization is not None:
                    logger.warning(
                        "[initialize_service] Quantization disabled: {}",
                        exc,
                    )
                    self.quantization = None
                else:
                    raise

            from acestep.model_downloader import get_checkpoints_dir
            env_ckpt = os.environ.get("ACESTEP_CHECKPOINTS_DIR")
            if env_ckpt:
                checkpoint_dir = str(get_checkpoints_dir())
            elif project_root:
                checkpoint_dir = os.path.join(project_root, "checkpoints")
            else:
                checkpoint_dir = str(get_checkpoints_dir())
            checkpoint_path = Path(checkpoint_dir)

            precheck_failure = self._ensure_models_present(
                checkpoint_path=checkpoint_path,
                config_path=config_path,
                prefer_source=prefer_source,
            )
            if precheck_failure is not None:
                self.model = None
                self.vae = None
                self.text_encoder = None
                self.text_tokenizer = None
                self.config = None
                self.silence_latent = None
                return precheck_failure

            self._sync_model_code_if_needed(config_path, checkpoint_path)

            model_path = os.path.join(checkpoint_dir, config_path)
            self._load_main_model_from_checkpoint(
                model_checkpoint_path=model_path,
                device=resolved_device,
                use_flash_attention=use_flash_attention,
                compile_model=normalized_compile,
                quantization=self.quantization,
            )
            vae_path = self._load_vae_model(
                checkpoint_dir=checkpoint_dir,
                device=resolved_device,
                compile_model=normalized_compile,
            )
            text_encoder_path = self._load_text_encoder_and_tokenizer(
                checkpoint_dir=checkpoint_dir,
                device=resolved_device,
            )

            mlx_dit_status, mlx_vae_status = self._initialize_mlx_backends(
                device=resolved_device,
                use_mlx_dit=use_mlx_dit,
                mlx_compile_requested=mlx_compile_requested,
            )

            status_msg = self._build_initialize_status_message(
                device=resolved_device,
                model_path=model_path,
                vae_path=vae_path,
                text_encoder_path=text_encoder_path,
                dtype=self.dtype,
                attention=getattr(self.config, "_attn_implementation", "eager"),
                compile_model=normalized_compile,
                mlx_compile_requested=mlx_compile_requested,
                offload_to_cpu=offload_to_cpu,
                offload_dit_to_cpu=offload_dit_to_cpu,
                quantization=self.quantization,
                mlx_dit_status=mlx_dit_status,
                mlx_vae_status=mlx_vae_status,
            )

            self.last_init_params = {
                "project_root": project_root,
                "config_path": config_path,
                "device": resolved_device,
                "use_flash_attention": use_flash_attention,
                "compile_model": normalized_compile,
                "offload_to_cpu": offload_to_cpu,
                "offload_dit_to_cpu": offload_dit_to_cpu,
                "quantization": self.quantization,
                "use_mlx_dit": use_mlx_dit,
                "prefer_source": prefer_source,
            }

            return status_msg, True
        except Exception as exc:
            self.model = None
            self.vae = None
            self.text_encoder = None
            self.text_tokenizer = None
            self.config = None
            self.silence_latent = None
            error_msg = f"Error initializing model: {str(exc)}\\n\\nTraceback:\\n{traceback.format_exc()}"
            logger.exception(error_msg)
            return error_msg, False
"""

try:
    with open(file_path, "w") as f:
        f.write(patched_code)
    print(f"✅ Successfully overwrote:\n{file_path}\n\nThe float32 environment patch is now active!")
except Exception as e:
    print(f"❌ Failed to overwrite the file. Error: {e}")

✅ Successfully overwrote:
/content/ACE-Step-1.5/acestep/core/generation/handler/init_service_orchestrator.py

The float32 environment patch is now active!


In [7]:
# @title Patch audio_utils.py (Fix Issue #1073)
import os
import requests

# Temporary Fix RuntimeError: Could not load libtorchcodec.
# https://github.com/ace-step/ACE-Step-1.5/issues/1073
local_file_path = '/content/ACE-Step-1.5/acestep/audio_utils.py'

# The URL for the patched version
patch_url = "https://raw.githubusercontent.com/dusterbloom/ACE-Step-1.5/3ef649664027d44a9bac7e7d8013cbab5bc83f63/acestep/audio_utils.py"

try:
    print(f"📡 Downloading patch from: {patch_url}")
    response = requests.get(patch_url)
    response.raise_for_status() # Ensure the download was successful
    
    # Write the patched content
    with open(local_file_path, 'w') as f:
        f.write(response.text)
    
    print(f"✅ Successfully patched: {local_file_path}")
except Exception as e:
    print(f"❌ Failed to apply patch: {e}")

📡 Downloading patch from: https://raw.githubusercontent.com/dusterbloom/ACE-Step-1.5/3ef649664027d44a9bac7e7d8013cbab5bc83f63/acestep/audio_utils.py
✅ Successfully patched: /content/ACE-Step-1.5/acestep/audio_utils.py


In [10]:
# @title Step 4: Launch Music Generator
%cd /content/ACE-Step-1.5

import os
os.environ["TORCH_CUDNN_SDPA_ENABLED"] = "0"
os.environ["ATTN_BACKEND"] = "eager"
os.environ["ACESTEP_ATTENTION_BACKEND"] = "eager"
os.environ["ACESTEP_DTYPE"] = "float32"
os.environ["MPLBACKEND"] = "agg"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["PATH"] = os.path.expanduser("~/.local/bin") + ":" + os.environ["PATH"]

LAUNCHER = r'''
import sys
import os
import importlib
import pkgutil
import traceback

os.environ["MPLBACKEND"] = "agg"


def run_custom_ui(dit, llm, **kwargs):
    import gradio as gr
    import random
    from acestep.inference import generate_music, GenerationParams, GenerationConfig, format_sample

    try:
        from acestep.gradio_ui.ui_components import EXAMPLE_POOL
        print(f"Loaded {len(EXAMPLE_POOL)} examples from original UI")
    except Exception:
        try:
            from acestep.gradio_ui.examples import get_examples
            EXAMPLE_POOL = get_examples()
            print(f"Loaded {len(EXAMPLE_POOL)} examples")
        except Exception:
            EXAMPLE_POOL = [
                {
                    "caption": "aggressive, high-energy heavy metal, dual heavily distorted guitars, chugging riffs, powerful driving drums, male vocals forceful raspy, melodic technical guitar solo, fast runs expressive bends, clean powerful production, anthemic rebellious",
                    "lyrics": (
                        "[Intro - Guitar Riff]\n\n"
                        "[Verse 1]\nRip the script, break it mold\nShatter chains, dare the bold\n"
                        "Took your crown, you can't ignore\nFreedom's knock on every door\n\n"
                        "[Chorus]\nBreak the chains, let it pour\nShout it loud, beg no more\n"
                        "Burn the rules, end the war\nWe're alive, can't ignore\nCan't ignore\n\n"
                        "[Verse 2]\nTear it down, brick by brick\nEvery lie, every trick\n"
                        "Skies on fire, won't obey\nChaos calls, don't delay\n\n"
                        "[Chorus]\nBreak the chains, let it pour\nShout it loud, beg no more\n"
                        "Burn the rules, end the war\nWe're alive, can't ignore\nCan't ignore\n\n"
                        "[Guitar Solo]\n\n"
                        "[Verse 3]\nEchoes scream, hearts collide\nNo surrender, no divide\n"
                        "Wolves are howling, hear their cry\nWe rise up, won't comply\n\n"
                        "[Chorus]\nBreak the chains, let it pour\nShout it loud, beg no more\n"
                        "Burn the rules, end the war\nWe're alive, can't ignore\nCan't ignore\n\n"
                        "[Outro - Guitar Riff]\n[abrupt silence]"
                    ),
                    "language": "en",
                },
                {
                    "caption": "female vocals, rap, modern, hip hop, Indian fusion, whispered, plucked synth melody, drums of earth",
                    "lyrics": (
                        "[Intro - Plucked Synth Melody]\n\n"
                        "[Verse 1]\nThe sun melts over the endless plain\nA mirage dances like a golden chain\n"
                        "Bare feet trace stories in the cracked dry land\nWho holds the map in this empty game?\n\n"
                        "[Chorus]\nDesert echoes\nThey call my name\nDrums of the earth, they beat the same\n"
                        "Step to the rhythm, no one to blame\nDesert echoes\nDesert echoes\n\n"
                        "[Instrumental Break - Plucked Synth Melody]\n\n"
                        "[Verse 2]\nA lone wind whispers secrets untold\nStars blink brighter than the city's gold\n"
                        "My shadow stretches long in the moonlit fold\nThis is my world, fierce and bold\n\n"
                        "[Bridge]\nDo you hear it? The silence screams\nDid you see it? The shifting dreams\n"
                        "[Vocal chop: 'dreams']\n\n"
                        "[Chorus]\nDesert echoes\nThey call my name\nDrums of the earth, they beat the same\n"
                        "Step to the rhythm, no one to blame\nDesert echoes\n\n"
                        "[Plucked synth melody fades out]"
                    ),
                    "language": "en",
                },
                {
                    "caption": "romantic active synthpop anthem, joyful energy, irresistible charm, bright engaging female vocals, feelgood danceable",
                    "lyrics": (
                        "Midnight talks with my regrets,\nWhiskey lips and cigarettes.\n"
                        "Promises like shattered glass,\nCut me deep but never last.\n\n"
                        "I danced with fire, played the fool,\nBroke the rules and called it cool.\n"
                        "Now the mirror wont lie\nI dont recognize these eyes.\n\n"
                        "I was right, I was left\nCaught between the lines, no regrets\n"
                        "I was wrong, I was left behind\nLike a shadow fading with the light\n\n"
                        "I've been up, but mostly down\nChasing dreams that hit the ground\n"
                        "Every high just slips away\nLeaving echoes of my name\n\n"
                        "I was right, I was left,\nTorn in two with every step.\n"
                        "I was wrong, I was left behind,\nAnother love I couldnt find.\n\n"
                        "Ive been up, but mostly down,\nChasing highs that let me drown.\n"
                        "I was right, I was left,\nNow Im at a new low."
                    ),
                    "language": "en",
                },
                {
                    "caption": "smooth jazzy lo-fi hip-hop, gentle piano melody, relaxed steady drum machine, warm round bassline, duet clear melodic female and smooth conversational male vocals, tasteful melodic saxophone fills, late-night atmosphere, extended instrumental outro, expressive improvisational sax solo",
                    "lyrics": (
                        "[Intro: Piano & Saxophone]\n\n"
                        "[Verse 1: Female Vocal]\n月光落进窗\n像是起色的那杯\n你还在我的心底旋转\n抛开所有烦忧\n"
                        "这爱像流星坠落\n在夜空闪耀\n无需解答 心动是唯一的解药\n\n"
                        "[Verse 2: Male Vocal]\n你的眼神像迷宫\n我徘徊其中\n每个音符都带了心脏的轰鸣\n"
                        "触碰你的温度就像电流穿心\n无法抗拒你的爱把我拉入梦境\n\n"
                        "[Chorus: Duet]\n心跳随着你狂奔\n如海潮汹涌\n爱的节奏翻滚着\n像梦境中梦\n"
                        "你的名字是旋律\n旋律中成风\n每一拍都诉说爱无声的惊动\n\n"
                        "[Instrumental Break: Saxophone Solo]\n\n"
                        "[Verse 3: Male Vocal]\n黑夜像个舞台\n我们点了彩灯\n节拍跟随心跳\n默契融成一吻\n"
                        "别让夜晚停下来\n爱永远在认真\n每一秒都像歌\n开启奇妙人生\n\n"
                        "[Chorus: Duet]\n心跳随着你狂奔\n如海潮汹涌\n爱的节奏翻滚着\n像梦境中梦\n"
                        "你的名字是旋律\n旋律中成风\n每一拍都诉说爱无声的惊动\n\n"
                        "[Outro: Duet & Saxophone]\n[Saxophone melody fades out]"
                    ),
                    "language": "zh",
                },
                {
                    "caption": "explosive high-energy K-pop EDM, relentless four-on-the-floor beat, pulsing synth bassline, bright piano melody, shimmering arpeggiated synths, powerful clear male lead vocal, anthemic melody, energetic ad-libs hype-man shouts",
                    "lyrics": (
                        "[Intro]\n[Synth arpeggio and pads]\n\n"
                        "[Verse 1]\n[ko] hwangholhan i jilseo-e neon ppajyeo\n[en] (oh yeah)\n"
                        "[ko] lideum wi-e sinhoneul jilleo\n[en] (oh, let's go)\n\n"
                        "[Chorus]\n[ko] oelo-un i jilseo gip-eojyeo\n[en] (yeah, yeah)\n"
                        "[ko] lideum wi-e him-eul jwi-yeo\n[en] (let's go)\n\n"
                        "[Instrumental Break]\n[Synth lead melody with vocal chops]\n\n"
                        "[Outro]\n[abrupt silence]"
                    ),
                    "language": "ko",
                },
                {
                    "caption": "rap, minimal, spoken word, dark, minimal beats heavy basslines, sharp snares, female vocals, introspective raw energy",
                    "lyrics": (
                        "[Intro]\n[Music box synth]\n[whispered] Yeah\n\n"
                        "[Verse 1]\n這不是我的街 我的根\n地下的 pum pum 像心跳\n敲響那律動 我的兄弟\n\n"
                        "[Chorus]\n不是抱怨 只是看清\n在這迷幻的夜裡思考人生的意義\n\n"
                        "[Outro]\n[Beat fades]"
                    ),
                    "language": "zh",
                },
                {
                    "caption": "driving post-punk, layered electric guitars, male lead vocal angsty strained, anthemic shouted chorus",
                    "lyrics": (
                        "[Intro - Guitar]\n\n"
                        "[Verse 1]\nUnder neon lights, they flicker and fade\nLost in the hum of this restless parade\n\n"
                        "[Chorus]\nOh, shadows of neon\nWhere do you run?\n\n"
                        "[Guitar Solo]\n\n"
                        "[Outro - fade]"
                    ),
                    "language": "en",
                },
                {
                    "caption": "classic country-folk ballad, steady acoustic guitar, warm male vocal gentle twang, plaintive harmonica",
                    "lyrics": (
                        "[Intro - Acoustic Guitar]\n\n"
                        "[Verse 1]\nFound you standing in the river's roar\nHeartbeats racing like never before\n\n"
                        "[Chorus]\nI'll ride these heartstrings all night long\nSing our song where we belong\n\n"
                        "[Outro - Harmonica]"
                    ),
                    "language": "en",
                },
                {
                    "caption": "Funk, Soul, Groove, Male Vocals, Slap Bass, Horn Section, 1970s, Party Energy",
                    "lyrics": (
                        "[Drum Break]\nGet on up!\n\n"
                        "[Verse 1]\nI feel the rhythm moving in my feet\n\n"
                        "[Chorus]\nGive it up, turn it loose!\nWe got the funk!"
                    ),
                    "language": "en",
                },
                {
                    "caption": "symphonic metal epic duet, female soprano male baritone, orchestral strings, operatic choirs",
                    "lyrics": (
                        "[Intro - Orchestral]\n\n"
                        "[Verse 1 - Female]\nI see you walking through the ash\n\n"
                        "[Chorus - Duet]\nWe stand upon the edge of the night\n\n"
                        "[Outro - fade to black]"
                    ),
                    "language": "en",
                },
            ]
            print(f"Loaded {len(EXAMPLE_POOL)} fallback examples")

    LANG = {
        "en": "🇬🇧 English", "zh": "🇨🇳 Chinese", "ja": "🇯🇵 Japanese", "ko": "🇰🇷 Korean",
        "es": "🇪🇸 Spanish", "fr": "🇫🇷 French", "de": "🇩🇪 German", "it": "🇮🇹 Italian",
        "pt": "🇵🇹 Portuguese", "ru": "🇷🇺 Russian", "ar": "🇸🇦 Arabic", "hi": "🇮🇳 Hindi",
        "tr": "🇹🇷 Turkish", "pl": "🇵🇱 Polish", "nl": "🇳🇱 Dutch", "sv": "🇸🇪 Swedish",
        "no": "🇳🇴 Norwegian", "da": "🇩🇰 Danish", "fi": "🇫🇮 Finnish", "el": "🇬🇷 Greek",
        "he": "🇮🇱 Hebrew", "th": "🇹🇭 Thai", "vi": "🇻🇳 Vietnamese", "id": "🇮🇩 Indonesian",
        "ms": "🇲🇾 Malay", "tl": "🇵🇭 Tagalog", "ceb": "🇵🇭 Cebuano", "uk": "🇺🇦 Ukrainian",
        "cs": "🇨🇿 Czech", "hu": "🇭🇺 Hungarian", "ro": "🇷🇴 Romanian", "bg": "🇧🇬 Bulgarian",
        "hr": "🇭🇷 Croatian", "sr": "🇷🇸 Serbian", "sk": "🇸🇰 Slovak", "sl": "🇸🇮 Slovenian",
        "lt": "🇱🇹 Lithuanian", "lv": "🇱🇻 Latvian", "et": "🇪🇪 Estonian", "fa": "🇮🇷 Persian",
        "ur": "🇵🇰 Urdu", "bn": "🇧🇩 Bengali", "ta": "🇮🇳 Tamil", "te": "🇮🇳 Telugu",
        "mr": "🇮🇳 Marathi", "gu": "🇮🇳 Gujarati", "kn": "🇮🇳 Kannada", "ml": "🇮🇳 Malayalam",
        "pa": "🇮🇳 Punjabi", "sw": "🇰🇪 Swahili", "af": "🇿🇦 Afrikaans",
    }

    def lang_code(display_name):
        return {v: k for k, v in LANG.items()}.get(display_name, "en")

    def detect_language(lyrics):
        if not lyrics or lyrics.strip() == "[inst]":
            return "en"
        text = lyrics.lower()
        if any(ch in text for ch in ['星', '月', '爱', '梦', '心']):
            return "zh"
        if any(ch in text for ch in ['の', 'を', 'に', 'は', 'が']):
            return "ja"
        if any(ch in text for ch in ['이', '그', '저', '를', '은']):
            return "ko"
        if any(w in text for w in ['esta', 'amor', 'noche']):
            return "es"
        if any(ch in text for ch in ['ت', 'ن', 'م', 'ل', 'ع']):
            return "ar"
        if any(ch in text for ch in ['ฉ', 'เ', 'า', 'ม', 'ก']):
            return "th"
        if any(w in text for w in ['eg', 'og', 'går']):
            return "no"
        return "en"

    PROJECT_ROOT = "/content/ACE-Step-1.5"
    OUTPUT_DIR = os.path.join(PROJECT_ROOT, "gradio_outputs")
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    def wrap_init_engine(p=gr.Progress()):
        p(0.1, desc="Checking GPU...")
        print("Initializing Engine...")
        init_kwargs = {
            "project_root": PROJECT_ROOT,
            "config_path": "acestep-v15-turbo",
            "device": "cuda",
            "offload_to_cpu": True,
            "compile_model": False,
            "use_flash_attention": False,
        }
        try:
            p(0.3, desc="Loading DiT...")
            status, ok = dit.initialize_service(**init_kwargs)
            if not ok:
                raise RuntimeError(f"DiT Init Failed: {status}")
            print("DiT loaded")
        except Exception as e:
            raise gr.Error(f"Failed to initialize DiT: {e}")

        p(0.5, desc="Checking LLM model...")
        lm_model_name = "acestep-5Hz-lm-1.7B"
        lm_checkpoint_dir = os.path.join(PROJECT_ROOT, "checkpoints")
        lm_model_dir = os.path.join(lm_checkpoint_dir, lm_model_name)
        lm_loaded = False

        if not os.path.exists(lm_model_dir):
            print(f"Downloading {lm_model_name} from HuggingFace...")
            p(0.55, desc=f"Downloading {lm_model_name}...")
            try:
                from huggingface_hub import snapshot_download
                snapshot_download(
                    repo_id=f"ACE-Step/{lm_model_name}",
                    local_dir=lm_model_dir,
                    local_dir_use_symlinks=False,
                )
                print(f"Downloaded {lm_model_name}")
            except Exception as dl_e:
                print(f"Failed to download LLM: {dl_e}")

        p(0.7, desc="Loading LLM...")
        try:
            lm_status, lm_ok = llm.initialize(
                checkpoint_dir=lm_checkpoint_dir,
                lm_model_path=lm_model_name,
                backend="pt",
                device="cuda",
                offload_to_cpu=True,
            )
            if lm_ok:
                lm_loaded = True
                print("LLM loaded successfully!")
            else:
                print(f"LLM init returned: {lm_status}")
        except Exception as e:
            print(f"LLM failed: {e}")
            traceback.print_exc()

        p(1.0, desc="Ready!")
        suffix = " + LLM (CoT)" if lm_loaded else " (DiT only, no LLM)"
        return f"✅ Engine Ready!{suffix}", gr.update(interactive=True), gr.update(visible=False)

    def randomize_example():
        example = random.choice(EXAMPLE_POOL)
        caption = example.get("caption", "")
        lyrics = example.get("lyrics", "[inst]")
        lang_code_val = example.get("language", detect_language(lyrics))
        lang_display = LANG.get(lang_code_val, LANG["en"])
        return caption, lyrics, lang_display

    def gen_music_ui(tags, lyrics, dur, lang, ntracks, think, seed, bpm, key, ts, steps, guide, instrumental, p=gr.Progress()):
        if not tags:
            raise gr.Error("Style/Tags required!")
        final_lyrics = "[inst]" if instrumental else (lyrics or "[inst]")
        s, r = -1, True
        if seed and str(seed).strip() not in ["", "-1"]:
            try:
                s, r = int(seed), False
            except ValueError:
                pass
        params = GenerationParams(
            caption=tags,
            lyrics=final_lyrics,
            thinking=think,
            duration=float(dur),
            vocal_language=lang_code(lang),
            inference_steps=int(steps),
            guidance_scale=float(guide),
            seed=s,
            bpm=int(bpm) if bpm else None,
            keyscale=key,
            timesignature=ts,
            task_type="text2music",
        )
        config = GenerationConfig(batch_size=int(ntracks), use_random_seed=r, audio_format="mp3")
        p(0.2, desc="Generating...")
        res = generate_music(dit_handler=dit, llm_handler=llm, params=params, config=config, save_dir=OUTPUT_DIR)
        if not res.success:
            raise gr.Error(res.error)
        paths = [a.get("path") for a in res.audios if isinstance(a, dict) and a.get("path")]
        return (
            paths[0] if paths else None,
            paths[1] if len(paths) > 1 else None,
            f"Generated {len(paths)} track(s)",
        )

    def gen_cover_ui(ref, tags, lyr, strength, dur, lang, n, p=gr.Progress()):
        if not ref:
            raise gr.Error("Upload audio!")
        params = GenerationParams(
            caption=tags,
            lyrics=lyr or "[inst]",
            thinking=True,
            duration=float(dur),
            vocal_language=lang_code(lang),
            task_type="cover",
            reference_audio=ref,
            audio_cover_strength=float(strength),
        )
        config = GenerationConfig(batch_size=int(n), audio_format="mp3")
        p(0.2, desc="Generating cover...")
        res = generate_music(dit_handler=dit, llm_handler=llm, params=params, config=config, save_dir=OUTPUT_DIR)
        if not res.success:
            raise gr.Error(res.error)
        paths = [a.get("path") for a in res.audios if isinstance(a, dict) and a.get("path")]
        return (
            paths[0] if paths else None,
            paths[1] if len(paths) > 1 else None,
            f"Generated {len(paths)} cover(s)",
        )

    def gen_repaint_ui(src, tags, lyr, start, end, strength, dur, lang, p=gr.Progress()):
        if not src:
            raise gr.Error("Upload audio!")
        params = GenerationParams(
            caption=tags,
            lyrics=lyr or "[inst]",
            thinking=True,
            duration=float(dur),
            vocal_language=lang_code(lang),
            task_type="repaint",
            src_audio=src,
            repainting_start=float(start),
            repainting_end=float(end),
            audio_cover_strength=float(strength),
        )
        config = GenerationConfig(batch_size=1, audio_format="mp3")
        p(0.2, desc="Repainting...")
        res = generate_music(dit_handler=dit, llm_handler=llm, params=params, config=config, save_dir=OUTPUT_DIR)
        if not res.success:
            raise gr.Error(res.error)
        paths = [a.get("path") for a in res.audios if isinstance(a, dict) and a.get("path")]
        return paths[0] if paths else None, "Repaint complete"

    def format_ui(cap, lyr):
        res = format_sample(llm_handler=llm, caption=cap, lyrics=lyr)
        if not res.success:
            raise gr.Error(res.error)
        return res.caption, res.lyrics, f"BPM: {res.bpm} Key: {res.keyscale}"

    with gr.Blocks(theme=gr.themes.Soft(), title="ACE-Step 1.5 Music Studio") as demo:
        init_status = gr.Textbox(
            value="Engine not started. Click Initialize below.",
            label="System Status",
            interactive=False,
        )
        init_btn = gr.Button("Initialize Engine (Required First Step)", variant="primary", size="lg")

        with gr.Group(visible=True):
            with gr.Tabs():
                with gr.Tab("Create"):
                    tags = gr.Textbox(
                        label="Music Style & Tags",
                        placeholder="e.g., upbeat pop, electronic, female vocals...",
                        lines=2,
                    )
                    with gr.Row():
                        lang = gr.Dropdown(list(LANG.values()), value="🇬🇧 English", label="Language (50+ supported)")
                        instrumental = gr.Checkbox(False, label="Instrumental Only")
                    lyr = gr.Textbox(
                        label="Lyrics",
                        lines=6,
                        placeholder="[verse]\nYour lyrics...\n\nClick the submit button for a random example",
                        submit_btn="Random Example",
                    )
                    with gr.Row():
                        dur = gr.Slider(minimum=10, maximum=480, step=10, value=60, label="Duration (sec)")
                        ntracks = gr.Slider(minimum=1, maximum=4, step=1, value=2, label="Tracks")
                    seed = gr.Number(value=-1, label="Seed (-1 = random, each track unique)", precision=0)
                    with gr.Accordion("Advanced Settings", open=False):
                        with gr.Row():
                            bpm = gr.Number(0, label="BPM (0 = auto)", precision=0)
                            key = gr.Dropdown(
                                ["", "C major", "D major", "E major", "F major", "G major",
                                 "A major", "B major", "C minor", "D minor", "E minor"],
                                label="Key",
                                value="",
                            )
                            ts = gr.Dropdown(["", "4/4", "3/4", "6/8"], label="Time Signature", value="")
                        with gr.Row():
                            steps = gr.Slider(minimum=4, maximum=50, step=1, value=30, label="Steps")
                            guide = gr.Slider(minimum=1, maximum=15, step=0.5, value=8.0, label="Guidance")
                    think = gr.Checkbox(True, label="Thinking Mode")
                    btn = gr.Button("Generate Music", variant="primary", size="lg", interactive=False)
                    with gr.Row():
                        out1 = gr.Audio(label="Track 1")
                        out2 = gr.Audio(label="Track 2")
                    out_info = gr.Textbox(label="Generation Info", interactive=False)
                    lyr.submit(randomize_example, outputs=[tags, lyr, lang])
                    btn.click(
                        gen_music_ui,
                        [tags, lyr, dur, lang, ntracks, think, seed, bpm, key, ts, steps, guide, instrumental],
                        [out1, out2, out_info],
                    )

                with gr.Tab("Cover / Remix"):
                    gr.Markdown("### Upload audio to transform into a new style")
                    ref = gr.Audio(label="Reference Audio", type="filepath")
                    with gr.Row():
                        ctags = gr.Textbox(label="Target Style", placeholder="rock, energetic, electric guitar...")
                        cstr = gr.Slider(minimum=0, maximum=1, step=0.05, value=0.5, label="Cover Strength")
                    clyr = gr.Textbox(label="New Lyrics (Optional)", lines=4, placeholder="Leave empty to keep original")
                    cbtn = gr.Button("Generate Cover", variant="primary", size="lg", interactive=False)
                    with gr.Row():
                        cout1 = gr.Audio(label="Cover 1")
                        cout2 = gr.Audio(label="Cover 2")
                    cout_info = gr.Textbox(label="Info", interactive=False)
                    cbtn.click(gen_cover_ui, [ref, ctags, clyr, cstr, dur, lang, ntracks], [cout1, cout2, cout_info])

                with gr.Tab("Repaint"):
                    gr.Markdown("### Regenerate a specific section")
                    rsrc = gr.Audio(label="Source Audio", type="filepath")
                    rtags = gr.Textbox(label="Repaint Style", placeholder="jazz, smooth, saxophone...")
                    rlyr = gr.Textbox(label="Section Lyrics", lines=3)
                    with gr.Row():
                        rstart = gr.Number(0, label="Start (seconds)", precision=1)
                        rend = gr.Number(-1, label="End (-1 = end)", precision=1)
                        rstr = gr.Slider(minimum=0, maximum=1, step=0.05, value=0.7, label="Strength")
                    rbtn = gr.Button("Repaint Region", variant="primary", size="lg", interactive=False)
                    rout_audio = gr.Audio(label="Result")
                    rout_info = gr.Textbox(label="Info", interactive=False)
                    rbtn.click(gen_repaint_ui, [rsrc, rtags, rlyr, rstart, rend, rstr, dur, lang], [rout_audio, rout_info])

                with gr.Tab("Format Lyrics"):
                    gr.Markdown("### Auto-format lyrics and detect metadata")
                    fcap = gr.Textbox(label="Style", placeholder="pop, upbeat...")
                    flyr = gr.Textbox(label="Raw Lyrics", lines=6, placeholder="Your unformatted lyrics...")
                    fbtn = gr.Button("Format & Enhance", variant="primary", size="lg", interactive=False)
                    fcap_out = gr.Textbox(label="Enhanced Style", interactive=False)
                    flyr_out = gr.Textbox(label="Formatted Lyrics", lines=6, interactive=False)
                    fmeta_out = gr.Textbox(label="Detected Metadata", interactive=False)
                    fbtn.click(format_ui, [fcap, flyr], [fcap_out, flyr_out, fmeta_out])

        init_btn.click(
            wrap_init_engine,
            outputs=[init_status, btn, init_btn],
        ).then(
            lambda: [gr.update(interactive=True)] * 3,
            outputs=[cbtn, rbtn, fbtn],
        )

    return demo


# --- Module patching ---

import acestep

found = False
target_func = "create_gradio_interface"
path = getattr(acestep, "__path__", [])

for _, name, _ in pkgutil.walk_packages(path, prefix="acestep."):
    try:
        mod = importlib.import_module(name)
        if hasattr(mod, target_func):
            print(f"Patching {target_func} in {mod.__name__}...")

            def wrapper(*args, **kwargs):
                dit = next((a for a in args if "AceStepHandler" in str(type(a))), args[0] if args else None)
                llm = next((a for a in args if "LLMHandler" in str(type(a))), args[1] if len(args) > 1 else None)
                return run_custom_ui(dit, llm, **kwargs)

            setattr(mod, target_func, wrapper)
            found = True
            break
    except ImportError:
        pass

if not found:
    print("Primary patch failed, trying fallback...")
    try:
        import acestep.gradio_ui.interface
        acestep.gradio_ui.interface.create_gradio_interface = lambda *a, **k: run_custom_ui(a[0], a[1], **k)
        print("Fallback patch successful!")
    except Exception:
        print("Failed to patch UI.")

# --- Entry point ---

try:
    import tomllib
except ImportError:
    import tomli as tomllib

with open("pyproject.toml", "rb") as f:
    pyproject = tomllib.load(f)

# NOTE: Do not pass --lm_model_path here; model download and init is handled
# inside wrap_init_engine() when the user clicks "Initialize Engine".
sys.argv = ["acestep", "--share"]

scripts = pyproject.get("project", {}).get("scripts", {})
entry = scripts.get("acestep", "")

if entry and ":" in entry:
    mod_path, func = entry.rsplit(":", 1)
    print(f"Running entry point: {mod_path}:{func}")
    m = importlib.import_module(mod_path)
    getattr(m, func)()
else:
    import runpy
    runpy.run_module("acestep", run_name="__main__")
'''

with open("/content/ACE-Step-1.5/custom_launch.py", "w") as f:
    f.write(LAUNCHER)
print("📝 Custom launcher written")
print("🎵 Launching AIQuest Academy Music Studio...")
print("🔗 Watch for the public Gradio URL!\n")

!MPLBACKEND=agg uv run python custom_launch.py

/content/ACE-Step-1.5
📝 Custom launcher written
🎵 Launching AIQuest Academy Music Studio...
🔗 Watch for the public Gradio URL!

Error in sitecustomize; set PYTHONVERBOSE for traceback:
ModuleNotFoundError: No module named 'wrapt'
Loaded configuration from /content/ACE-Step-1.5/.env.example (fallback)
2026-04-09 03:39:15.759 | DEBUG    | acestep.core.generation.handler.init_service_memory_basic:_apply_malloc_mmap_threshold:50 - [memory] Set M_MMAP_THRESHOLD=131072 for immediate OS reclaim of large frees
2026-04-09 03:39:21.800 | WARNING  | acestep.training.trainer:<module>:40 - bitsandbytes not installed. Using standard AdamW.
Patching create_gradio_interface in acestep.acestep_v15_pipeline...
Running entry point: acestep.acestep_v15_pipeline:main
2026-04-09 03:39:23.566 | INFO     | acestep.gpu_config:get_gpu_memory_gb:578 - CUDA GPU detected: Tesla T4 (14.6 GB)

GPU Configuration Detected:
  GPU Memory: 14.56 GB
  Configuration Tier: tier5
  Max Duration (with LM): 480s (8 min)
  Max 